# A2C et A2C-GAE sur HalfCheetah : l'actor-critic pour le controle continu

**Algorithmes** : A2C (Advantage Actor-Critic) et A2C-GAE (avec Generalized Advantage Estimation), construits sur une base commune puis compares  
**Environnement** : HalfCheetah-v5 (MuJoCo, Gymnasium)  
**Papiers** : [Mnih et al., 2016](https://arxiv.org/abs/1602.01783) (A3C/A2C) ; [Schulman et al., 2016](https://arxiv.org/abs/1506.02438) (GAE)

| Propriete | A2C | A2C-GAE |
|-----------|-----|----------|
| Model-free | oui | oui |
| On-policy | oui | oui |
| Actor-Critic | oui | oui |
| TD bootstrapping | oui (n-step) | oui (exponentiel) |
| Replay buffer | non | non |
| Politique gaussienne | oui | oui |
| Estimateur d'avantage | n-step returns | GAE($\lambda$) |
| Biais-variance | reglage fixe | reglage continu via $\lambda$ |

Dans le notebook precedent, on a construit REINFORCE et REINFORCE+baseline sur CartPole. Ces methodes sont **Monte Carlo** : elles attendent la fin de l'episode pour calculer les returns. Sur CartPole (200-500 pas), c'est acceptable. Sur HalfCheetah (1000 pas), avec des actions continues, on a besoin de quelque chose de plus puissant.

Ce notebook introduit **A2C** (Advantage Actor-Critic), qui resout les trois limites fondamentales de REINFORCE, et **A2C-GAE** qui affine l'estimation de l'avantage. Les deux algorithmes partagent tout sauf la facon de calculer l'avantage : c'est le **fork** central du notebook.

## De REINFORCE a Actor-Critic : trois problemes, trois solutions

### Les limites de REINFORCE

REINFORCE+baseline fonctionne bien sur CartPole. Mais il a trois problemes fondamentaux qui apparaissent sur des environnements plus serieux :

**Probleme 1 : les episodes longs de MuJoCo.**  
HalfCheetah a des episodes de 1000 pas. REINFORCE attend la fin de l'episode pour mettre a jour la politique. Pendant ces 1000 pas, l'agent explore sans rien apprendre. Sur CartPole (200 pas max), c'est tolerable. Sur HalfCheetah, c'est ruineux en efficacite.

**Probleme 2 : la variance Monte Carlo.**  
Le return $G_t = \sum_{k \geq t} \gamma^k r_{t+k}$ accumule du bruit sur tous les pas futurs. Avec 1000 pas, la variance est enorme. La normalisation et la baseline aident, mais ne suffisent pas.

**Probleme 3 : les actions continues.**  
CartPole a 2 actions discretes. HalfCheetah a 6 dimensions continues. On ne peut plus utiliser une distribution categorielle. Il faut une distribution gaussienne parametree par le reseau.

### Les trois solutions d'A2C

| Probleme | Solution A2C |
|----------|--------------|
| Episodes longs | **Rollout de n pas** : on met a jour apres $n$ pas, pas apres un episode entier |
| Variance MC | **Bootstrap TD** : $G_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n})$, moins de variance |
| Actions continues | **Politique gaussienne** : le reseau parametre $\mu_\theta(s)$, $\sigma$ est un parametre appris |

**Analogie.** REINFORCE attend la fin du match pour noter les joueurs. A2C note apres chaque mi-temps (n-step rollout). L'evaluation est moins complete mais beaucoup plus frequente, et l'entraineur peut corriger le tir bien plus vite.

### L'actor et le critic

A2C introduit explicitement deux roles :
- **L'actor** $\pi_\theta(a|s)$ : le reseau de politique qui choisit les actions. Il apprend a maximiser la recompense.
- **Le critic** $V_\phi(s)$ : le reseau de valeur qui evalue les etats. Il fournit une baseline pour reduire la variance et permet le bootstrapping.

Les deux s'entrainent ensemble, en boucle. Le critic evalue les performances de l'actor. L'actor utilise l'evaluation du critic pour ajuster sa politique. C'est la dynamique actor-critic.

## Nouvel environnement : HalfCheetah-v5

### Pourquoi changer d'environnement ?

CartPole est un excellent banc de test pour les concepts de base : petit, rapide, facile a visualiser. Mais il est **trop simple** pour les algorithmes actor-critic modernes. Q-learning et DQN le resolvent en quelques centaines d'episodes. REINFORCE le resolve correctement. Il ne presente aucun defi pour A2C.

HalfCheetah est un premier pas vers le **controle continu reel** : robots, vehicules autonomes, simulation physique. C'est l'environnement standard de benchmarking pour A2C, PPO, SAC, TD3. Maitriser HalfCheetah, c'est maitriser la fondation du RL moderne.

### Description physique

HalfCheetah est un robot 2D simule dans MuJoCo. Il ressemble a la moitie d'un guepard : un torse articule avec **deux pattes** (avant et arriere), chaque patte ayant **3 joints** (hanche, genou, cheville). Le robot est ancre lateralement — il ne peut que courir vers la droite.

L'objectif : **courir le plus vite possible vers la droite**.

### Espace d'observation (17 dimensions)

L'etat capture la configuration complete du robot :

| Indices | Variable | Description |
|---------|----------|-------------|
| 0 | `z_pos` | Position verticale du torse |
| 1 | `x_vel` | Vitesse horizontale (ce qu'on veut maximiser) |
| 2 | `z_angle` | Angle d'inclinaison du torse |
| 3-5 | angles des joints avant | Hanche, genou, cheville de la patte avant |
| 6-8 | angles des joints arriere | Hanche, genou, cheville de la patte arriere |
| 9-11 | velocites angulaires avant | Vitesses des 3 joints de la patte avant |
| 12-14 | velocites angulaires arriere | Vitesses des 3 joints de la patte arriere |
| 15-16 | velocites angulaires torse | Rotation et inclinaison du torse |

Soit 17 dimensions de nombres reels continus, sans bornes strictes.

### Espace d'actions (6 dimensions continues)

`Box(-1, 1, shape=(6,))` : 6 couples moteurs, un par joint :

| Dimension | Joint controlé |
|-----------|---------------|
| 0 | Hanche arriere |
| 1 | Genou arriere |
| 2 | Cheville arriere |
| 3 | Hanche avant |
| 4 | Genou avant |
| 5 | Cheville avant |

Chaque valeur est entre $-1$ et $+1$. Un couple positif pousse le joint dans une direction, negatif dans l'autre.

### Recompense

$$r_t = \underbrace{\text{forward\_reward}}_{{\approx v_x}} - \underbrace{\text{ctrl\_cost}}_{0.1 \cdot \|a_t\|^2}$$

L'agent est **recompense pour aller vite** et **penalise pour les grandes actions** (controle energetique). Pas de condition de terminaison : l'episode dure exactement 1000 pas (truncation uniquement). Pas de "game over" si le robot tombe.

### Pourquoi les actions continues changent tout

Avec CartPole, on avait 2 actions discretes. On pouvait faire $\arg\max Q(s, a)$, ou utiliser une distribution categorielle. Avec HalfCheetah, l'espace d'actions est $\mathbb{R}^6$ (en pratique $[-1,1]^6$ continu).

- $\arg\max$ : **impossible** sur un espace continu (infiniment d'actions possibles)
- $\varepsilon$-greedy : **impossible** (pas d'action discrete a explorer)
- Distribution categorielle : **impossible** (distribution sur un infini de valeurs)

**La seule solution** : parametrer une distribution continue. On choisit une gaussienne $\mathcal{N}(\mu_\theta(s), \sigma^2)$. Le reseau predit la moyenne $\mu$, et on echantillonne une action.

### Comparaison CartPole vs HalfCheetah

| Dimension | CartPole-v1 | HalfCheetah-v5 |
|-----------|-------------|----------------|
| Observation | 4 dims | 17 dims |
| Actions | 2 discretes | 6 continues |
| Type d'action | Categorielle | Gaussienne |
| Longueur episode | 200-500 pas | 1000 pas (fixe) |
| Condition de fin | Pendule tombe | Truncation uniquement |
| Difficulte | Jouet | Benchmark reel |
| Algorithme minimal | Q-learning | A2C / PPO |
| Score aleatoire | ~20 | ~-300 |
| Score bien entraine | 500 | 3000-6000 |

In [ ]:
import inspect
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from gymnasium.wrappers import RecordVideo
from torch.distributions import Normal

try:
    from IPython.display import Video, display
except ImportError:
    Video = None
    display = None

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["figure.dpi"] = 100
print(f"PyTorch : {torch.__version__}")
print(f"Gymnasium : {gym.__version__}")

try:
    env_test = gym.make("HalfCheetah-v5")
    env_test.close()
    print("MuJoCo : disponible")
except Exception as e:
    print(f"MuJoCo : ERREUR -> {e}")
    print("Installez avec : pip install gymnasium[mujoco]")


In [ ]:
# === Exploration detaillee de HalfCheetah ===
env = gym.make("HalfCheetah-v5")
obs, info = env.reset(seed=42)

print("=" * 50)
print("HalfCheetah-v5 : espace des observations")
print("=" * 50)
print(f"Forme       : {env.observation_space.shape}")
print(f"Type        : {env.observation_space.dtype}")
print(f"Bornes min  : {env.observation_space.low[:5]}... (tronquees)")
print(f"Bornes max  : {env.observation_space.high[:5]}... (tronquees)")
print(f"\nObservation initiale (17 dims) :")
labels = ["z_pos", "x_vel", "z_angle",
          "hip_f", "knee_f", "ankle_f",
          "hip_b", "knee_b", "ankle_b",
          "vel_hip_f", "vel_knee_f", "vel_ankle_f",
          "vel_hip_b", "vel_knee_b", "vel_ankle_b",
          "vel_torse_rot", "vel_torse_ang"]
for i, (label, val) in enumerate(zip(labels, obs)):
    print(f"  [{i:2d}] {label:15s} : {val:.4f}")

print("\n" + "=" * 50)
print("HalfCheetah-v5 : espace des actions")
print("=" * 50)
print(f"Forme       : {env.action_space.shape}")
print(f"Type        : {env.action_space.dtype}")
print(f"Bornes      : [{env.action_space.low[0]:.1f}, {env.action_space.high[0]:.1f}] pour chaque dimension")

# Episode aleatoire pour voir les recompenses
obs, _ = env.reset(seed=42)
total_reward, steps = 0.0, 0
rewards_random = []
for _ in range(200):  # 200 pas seulement pour la demo
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    rewards_random.append(reward)
    steps += 1
    if terminated or truncated:
        break

print(f"\nEpisode aleatoire (200 pas) :")
print(f"  Recompense totale : {total_reward:.2f}")
print(f"  Recompense moyenne par pas : {np.mean(rewards_random):.4f}")
print(f"  Min / Max par pas : {min(rewards_random):.4f} / {max(rewards_random):.4f}")
print(f"\nComparaison :")
print(f"  CartPole : obs=(4,), actions=2 discretes")
print(f"  HalfCheetah : obs=(17,), actions={env.action_space.shape[0]} continues")
env.close()

## La politique gaussienne

### De Categorical a Normal

Sur CartPole, on utilisait `Categorical(logits=...)` : le reseau produisait des logits, on appliquait un softmax pour obtenir des probabilites sur 2 actions discretes.

Sur HalfCheetah, l'espace d'actions est $[-1, 1]^6$ continu. On utilise une **distribution gaussienne multivariee diagonale** :

$$\pi(a \mid s; \theta) = \mathcal{N}(\mu_\theta(s),\ \text{diag}(\sigma^2))$$

Le reseau predit la **moyenne** $\mu_\theta(s) \in \mathbb{R}^6$ pour chaque dimension d'action. L'**ecart-type** $\sigma$ est un **vecteur de parametres appris independamment** (pas en sortie du reseau).

### Pourquoi $\log(\sigma)$ et pas $\sigma$ ?

On parametre $\log(\sigma)$ plutot que $\sigma$ directement. Deux raisons :

1. **Contrainte positive** : $\sigma > 0$ est requis pour une gaussienne valide. Avec $\log(\sigma) \in \mathbb{R}$, on a automatiquement $\sigma = \exp(\log(\sigma)) > 0$, sans contrainte explicite ni activation speciale.
2. **Gradients stables** : $\sigma$ peut varier sur plusieurs ordres de grandeur. $\log(\sigma)$ est dans une plage numeriquement raisonnable, les gradients sont plus stables.

### Pourquoi $\log(\sigma)$ est un parametre et pas une sortie du reseau ?

C'est la convention standard dans A2C et PPO. Raisonner en termes pratiques :
- Si log_std est en sortie du reseau, il depend de l'etat $s$. La variance d'exploration change selon la situation. C'est plus expressif mais plus difficile a stabiliser.
- Si log_std est un parametre global, l'exploration est uniforme sur tous les etats. Plus simple, plus stable, suffisant pour HalfCheetah.

### Log-probabilite pour une action multidimensionnelle

Pour une covariance diagonale, les 6 dimensions sont **independantes** :

$$\log \pi(a \mid s; \theta) = \sum_{d=1}^{6} \log \mathcal{N}(a_d;\ \mu_d,\ \sigma_d^2)$$

En code : `dist.log_prob(action).sum(dim=-1)`. Le `.sum(dim=-1)` additionne les log-probabilites sur les 6 dimensions d'action (produit de probabilites independantes = somme de log-probabilites).

### Pourquoi cette architecture ?

Le `GaussianActor` est un MLP assez classique pour MuJoCo : deux couches cachees de 256 neurones avec activation $\tanh$.

Deux couches cachees donnent au reseau assez de capacite pour approximer une politique non lineaire. HalfCheetah n'est pas visuellement complexe, mais sa dynamique physique l'est : une bonne action depend de la phase de course, des angles, des vitesses, et de la coordination entre les deux pattes. Une seule couche peut etre trop limitee; beaucoup plus de couches rendraient l'optimisation plus fragile pour peu de gain dans ce projet.

La largeur $256$ est un choix de reference courant dans les environnements MuJoCo. Ce n'est pas une constante theorique : $64$ ou $128$ peuvent marcher sur des runs courts, mais $256$ donne une capacite confortable tout en restant rapide.

On utilise $\tanh$ parce que les observations et les activations internes sont continues. Contrairement a ReLU, $\tanh$ borne les activations dans $[-1, 1]$, ce qui aide a eviter des sorties trop grandes et des updates trop brusques au debut de l'entrainement. ReLU serait possible, mais $\tanh$ est un choix conservateur et frequent pour les politiques continues actor-critic.

La tete $\mu_{head}$ est initialisee avec un petit gain ($0.01$). L'idee est que la politique commence avec des moyennes d'action proches de zero : elle ne pousse pas encore fortement les moteurs dans une direction arbitraire. L'exploration initiale vient surtout de l'ecart-type appris $\log(\sigma)$, initialise a $0$, donc $\sigma = \exp(0) = 1$.

Enfin, la gaussienne peut echantillonner des actions hors de $[-1, 1]$. On clippe donc les actions avant de les envoyer a l'environnement. Une alternative plus rigoureuse serait une gaussienne passee dans $\tanh$ avec correction du $\log_{prob}$, mais pour un A2C pedagogique, le clipping garde l'implementation lisible.

In [ ]:
def orthogonal_init(module, gain=1.0):
    """Initialisation orthogonale recommandee pour les reseaux actor-critic."""
    if isinstance(module, nn.Linear):
        nn.init.orthogonal_(module.weight, gain=gain)
        nn.init.zeros_(module.bias)


class GaussianActor(nn.Module):
    """Reseau de politique gaussienne pi(a|s; theta) pour actions continues.
    
    Sortie : mu_theta(s) in R^action_dim
    log_std : parametre appris independamment (pas en sortie du reseau)
    """

    def __init__(self, obs_dim: int, action_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.mu_head = nn.Linear(hidden_dim, action_dim)
        # log_std : parametre global, initialise a 0 (sigma=1 au depart)
        self.log_std = nn.Parameter(torch.zeros(action_dim))

        # Initialisation orthogonale (standard A2C/PPO)
        self.apply(lambda m: orthogonal_init(m, gain=np.sqrt(2)))
        orthogonal_init(self.mu_head, gain=0.01)  # petites sorties initiales

    def forward(self, x: torch.Tensor):
        """Retourne la distribution gaussienne pour un etat ou un batch d'etats."""
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        mu = self.mu_head(x)                    # (B, action_dim)
        std = self.log_std.exp().expand_as(mu)  # (B, action_dim)
        return Normal(mu, std)

    def get_action(self, obs: np.ndarray):
        """Echantillonne une action et retourne (action, log_prob)."""
        obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
        dist = self.forward(obs_t)
        action = dist.sample()                              # (1, action_dim)
        log_prob = dist.log_prob(action).sum(dim=-1)        # (1,) -> scalar
        return action.squeeze(0).numpy(), log_prob.squeeze(0)

    def evaluate(self, obs_t: torch.Tensor, actions_t: torch.Tensor):
        """Calcule log_prob et entropy pour un batch (training)."""
        dist = self.forward(obs_t)
        log_prob = dist.log_prob(actions_t).sum(dim=-1)     # (B,)
        entropy = dist.entropy().sum(dim=-1)                # (B,)
        return log_prob, entropy


# --- Demo ---
torch.manual_seed(42)
obs_dim, action_dim = 17, 6
actor = GaussianActor(obs_dim, action_dim, hidden_dim=256)

dummy_obs = np.zeros(obs_dim)
action, log_prob = actor.get_action(dummy_obs)

print(f"GaussianActor : {sum(p.numel() for p in actor.parameters()):,} parametres")
print(f"  dont log_std : {actor.log_std.numel()} parametres")
print(f"\nObservation shape : {dummy_obs.shape}")
print(f"Action echantillonnee : {action}")
print(f"  shape : {action.shape}")
print(f"  bornes : [{action.min():.3f}, {action.max():.3f}]")
print(f"log_prob : {log_prob.item():.4f}  (grad_fn={log_prob.grad_fn})")
print(f"\nlog_std initial : {actor.log_std.data}")
print(f"std initiale : {actor.log_std.exp().data}  (doit etre ~1.0)")

# Test batch
batch_obs = torch.randn(32, obs_dim)
batch_actions = torch.randn(32, action_dim)
lp, ent = actor.evaluate(batch_obs, batch_actions)
print(f"\nBatch : obs={batch_obs.shape}, actions={batch_actions.shape}")
print(f"  log_prob : {lp.shape}, entropy : {ent.shape}")

## Le reseau critique $V(s)$

### Role du critic dans A2C

Le critic $V_\phi(s)$ estime la **valeur d'etat** : la recompense totale esperee depuis l'etat $s$ en suivant la politique courante.

$$V_\phi(s) \approx V^\pi(s) = \mathbb{E}_{\pi}\left[\sum_{k \geq 0} \gamma^k r_{t+k} \mid s_t = s\right]$$

Il sert a deux choses :
1. **Bootstrap** : estimer la valeur d'un etat futur sans attendre la fin de l'episode. $G_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V_\phi(s_{t+n})$.
2. **Baseline** : centrer les avantages autour de zero. $A_t = G_t - V_\phi(s_t)$.

### Architecture : meme structure que l'actor

On utilise la meme architecture que l'actor (256 neurones, Tanh), sauf la couche de sortie qui produit un scalaire. Meme initialisation orthogonale. C'est la convention standard (separer actor et critic en deux reseaux independants, plutot que de partager les couches).

**Note.** Certaines implementations partagent les premieres couches entre actor et critic (economie de parametres). On garde des reseaux separes ici pour la clarte pedagogique.

In [ ]:
class CriticNetwork(nn.Module):
    """Reseau de valeur V(s; phi) : observation -> scalaire."""

    def __init__(self, obs_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.value_head = nn.Linear(hidden_dim, 1)

        # Initialisation orthogonale
        self.apply(lambda m: orthogonal_init(m, gain=np.sqrt(2)))
        orthogonal_init(self.value_head, gain=1.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        return self.value_head(x).squeeze(-1)  # (B,) ou () si scalaire


# --- Demo ---
torch.manual_seed(42)
critic = CriticNetwork(obs_dim=17, hidden_dim=256)

print(f"CriticNetwork : {sum(p.numel() for p in critic.parameters()):,} parametres")

# Valeur pour un etat unique
dummy_state = torch.zeros(17)
value = critic(dummy_state)
print(f"\nEtat shape : {dummy_state.shape}")
print(f"Valeur V(s) : {value.item():.4f}  (scalaire, grad_fn={value.grad_fn})")

# Valeur pour un batch
batch_states = torch.randn(32, 17)
values_batch = critic(batch_states)
print(f"\nBatch etats : {batch_states.shape} -> valeurs : {values_batch.shape}")
print(f"Valeurs min/max : {values_batch.min().item():.4f} / {values_batch.max().item():.4f}")
print('(avant entrainement : valeurs proches de 0 attendues avec init orthogonale)')

## Le RolloutBuffer

### On-policy : pas de replay buffer

A2C est **on-policy** : les donnees doivent etre generees par la **politique courante** $\pi_\theta$. Contrairement a DQN (off-policy avec replay buffer), on ne peut pas reutiliser des transitions generees par une ancienne politique.

La consequence : apres chaque mise a jour, le buffer est **vide** et on recommence la collecte avec la nouvelle politique.

### Pourquoi un rollout de n_steps et pas un episode complet ?

Sur HalfCheetah, un episode = 1000 pas. Si on attend la fin de l'episode comme REINFORCE :
- 1000 pas de collecte sans mise a jour : lent et inefficace
- Returns Monte Carlo avec 1000 pas de variance accumulee : tres bruite

Avec un rollout de `n_steps=2048` pas (qui peut couper un episode en plein milieu) :
- Mise a jour plus frequente
- Bootstrap : $G_t = r_t + \ldots + \gamma^{n-1} r_{t+n-1} + \gamma^n V(s_{t+n})$ avec $n < 1000$
- Variance reduite car on somme moins de pas avant de bootstrapper

### Structure du buffer

Pre-allouer des `np.ndarray` pour chaque type de donnee (observations, actions, recompenses, dones, valeurs). Bien plus efficace que des listes Python qui grandissent dynamiquement.

In [ ]:
class RolloutBuffer:
    """Buffer on-policy pre-alloue pour stocker n_steps transitions.
    
    Efface apres chaque mise a jour (on-policy).
    """

    def __init__(self, n_steps: int, obs_dim: int, action_dim: int):
        self.n_steps = n_steps
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        self.reset()

    def reset(self):
        """Remet le buffer a zero. Appelle apres chaque mise a jour."""
        self.observations = np.zeros((self.n_steps, self.obs_dim), dtype=np.float32)
        self.actions = np.zeros((self.n_steps, self.action_dim), dtype=np.float32)
        self.rewards = np.zeros(self.n_steps, dtype=np.float32)
        self.dones = np.zeros(self.n_steps, dtype=np.float32)
        self.values = np.zeros(self.n_steps, dtype=np.float32)
        self.log_probs = np.zeros(self.n_steps, dtype=np.float32)
        self.ptr = 0  # pointeur courant

    def add(self, obs, action, reward, done, value, log_prob):
        """Ajoute une transition au buffer."""
        assert self.ptr < self.n_steps, "Buffer plein"
        self.observations[self.ptr] = obs
        self.actions[self.ptr] = action
        self.rewards[self.ptr] = reward
        self.dones[self.ptr] = done
        self.values[self.ptr] = value
        self.log_probs[self.ptr] = log_prob
        self.ptr += 1

    @property
    def full(self):
        return self.ptr == self.n_steps

    def get_tensors(self):
        """Convertit le buffer en tenseurs PyTorch pour le training."""
        return (
            torch.tensor(self.observations, dtype=torch.float32),
            torch.tensor(self.actions, dtype=torch.float32),
            torch.tensor(self.rewards, dtype=torch.float32),
            torch.tensor(self.dones, dtype=torch.float32),
            torch.tensor(self.values, dtype=torch.float32),
            torch.tensor(self.log_probs, dtype=torch.float32),
        )


# --- Demo ---
buf = RolloutBuffer(n_steps=2048, obs_dim=17, action_dim=6)
print(f"RolloutBuffer(n_steps=2048, obs_dim=17, action_dim=6)")
mem = (buf.observations.nbytes + buf.actions.nbytes + buf.rewards.nbytes
       + buf.dones.nbytes + buf.values.nbytes + buf.log_probs.nbytes)
print(f"  Memoire totale : {mem / 1024:.1f} Ko")
print(f"  Plein : {buf.full}")

# Simuler l'ajout de quelques transitions
for i in range(5):
    buf.add(
        obs=np.random.randn(17).astype(np.float32),
        action=np.random.uniform(-1, 1, 6).astype(np.float32),
        reward=np.random.randn(),
        done=False,
        value=np.random.randn(),
        log_prob=np.random.randn(),
    )
print(f"  Apres 5 ajouts -> ptr={buf.ptr}, plein={buf.full}")

## TD bootstrapping et n-step returns

### Le probleme du biais-variance dans l'estimation du retour

Pour calculer la cible de l'avantage $A_t = G_t - V(s_t)$, on a besoin d'estimer $G_t$. On a trois options principales, et chacune fait un compromis different entre biais et variance :

**Monte Carlo (REINFORCE)** :
$$G_t^{MC} = \sum_{k=0}^{T-t} \gamma^k r_{t+k}$$
- **Biais** : zero (estimateur non biaise de $V^\pi(s_t)$)
- **Variance** : tres elevee (on somme $T-t$ termes bruits)

**TD(0)** :
$$G_t^{TD(0)} = r_t + \gamma V(s_{t+1})$$
- **Biais** : eleve (depend de la precision de $V$, qui est imparfaite)
- **Variance** : tres faible (un seul pas de recompense)

**N-step return** :
$$G_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n})$$
- **Biais** : modere (depend de $V$ mais seulement pour le terme bootstrappe)
- **Variance** : moderee (on somme $n$ termes, pas toute la trajectoire)

Le n-step return est le compromis choisi par A2C. Avec `n_steps=2048` et des episodes de 1000 pas, chaque rollout couvre au moins 2 episodes complets.

### Gestion des fins d'episode

Quand un episode se termine dans le rollout (done=True), le bootstrap s'arrete : $\gamma^n V(s_{t+n}) = 0$ si l'etat $s_{t+n}$ est terminal. Le flag `done` dans le buffer sert exactement a ca.

In [ ]:
def compute_returns(rewards: np.ndarray, dones: np.ndarray,
                    last_value: float, gamma: float = 0.99) -> np.ndarray:
    """Calcule les n-step returns bootstrappes depuis la fin du rollout.
    
    Args:
        rewards : recompenses du rollout (n_steps,)
        dones   : flags de fin d'episode (n_steps,)  -- 1.0 si terminal
        last_value : V(s_{t+n}) pour le bootstrap du dernier pas
        gamma   : facteur d'actualisation
    
    Returns:
        returns : estimations de G_t pour chaque pas (n_steps,)
    """
    n_steps = len(rewards)
    returns = np.zeros(n_steps, dtype=np.float32)
    G = last_value  # bootstrap depuis le dernier etat

    for t in reversed(range(n_steps)):
        # Si done=1, l'etat suivant est terminal : pas de bootstrap
        G = rewards[t] + gamma * G * (1.0 - dones[t])
        returns[t] = G

    return returns


# --- Demo sur un rollout fictif de 10 pas ---
np.random.seed(42)
n_demo = 10
rewards_demo = np.array([1.0, 0.5, -0.2, 1.0, 0.8, 0.3, 0.6, -0.1, 0.9, 0.4], dtype=np.float32)
# Episode termine au pas 4 (done=True)
dones_demo = np.array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0], dtype=np.float32)
last_value_demo = 2.5  # V(s_10) bootstrappe

returns_nstep = compute_returns(rewards_demo, dones_demo, last_value_demo, gamma=0.99)
returns_mc = compute_returns(rewards_demo, dones_demo, 0.0, gamma=0.99)  # MC : pas de bootstrap

print("Rollout fictif (10 pas, episode termine au pas 4) :")
print(f"  Recompenses : {rewards_demo}")
print(f"  Dones       : {dones_demo.astype(int)}")
print(f"  Last value  : {last_value_demo} (bootstrap)")
print(f"\n  Returns n-step : {returns_nstep.round(3)}")
print(f"  Returns MC     : {returns_mc.round(3)}")
print(f"\nObservation : apres done=1 au pas 4, le calcul repart de zero")
print(f"  -> G_4 = r_4 = {rewards_demo[4]:.1f} (done, pas de bootstrap)")
print(f"  -> G_9 = r_9 + gamma * last_value = {rewards_demo[9]:.1f} + 0.99 * {last_value_demo:.1f} = {rewards_demo[9] + 0.99 * last_value_demo:.3f}")
print(f"  -> calcule : {returns_nstep[9]:.3f}")

## L'entropy bonus

### Pourquoi encourager l'entropie ?

L'entropie d'une politique $\mathcal{H}[\pi(\cdot|s)] = -\mathbb{E}_{a \sim \pi}[\log \pi(a|s)]$ mesure l'alatoire de la politique. Une entropie elevee = politique tres exploratoire (distribution large). Une entropie faible = politique tres deterministe (distribution concentree).

En ajoutant un bonus d'entropie a la loss de politique, on encourage l'exploration :

$$\mathcal{L}_\pi = -\frac{1}{n}\sum_t \log\pi_\theta(a_t|s_t) \cdot A_t - c_H \cdot \mathcal{H}[\pi_\theta(\cdot|s_t)]$$

### Pourquoi entropy_coef=0.0 pour HalfCheetah ?

Pour une politique gaussienne, l'entropie est naturellement elevee a cause de $\sigma$. La distribution gaussienne avec $\sigma > 0$ est **deja stochastique par construction** : chaque action echantillonnee est differente. L'exploration n'est pas un probleme sur HalfCheetah.

Sur des politiques discretes (Atari, CartPole), le bonus d'entropie est crucial pour eviter que la politique ne converge trop vite vers une action unique. Sur les politiques continues avec $\sigma$ appris, c'est generalement inutile et peut meme ralentir la convergence.

## La loss totale A2C

### Les trois termes

$$\boxed{\mathcal{L} = \underbrace{-\frac{1}{n}\sum_t \log\pi_\theta(a_t|s_t) \cdot A_t}_{\text{policy loss}} + \underbrace{c_V \cdot \frac{1}{n}\sum_t (V_\phi(s_t) - G_t)^2}_{\text{value loss}} - \underbrace{c_H \cdot \frac{1}{n}\sum_t \mathcal{H}[\pi_\theta(\cdot|s_t)]}_{\text{entropy bonus}}}$$

**Policy loss** : maximise la log-probabilite des actions proportionnellement a leur avantage. Actions avec $A_t > 0$ (mieux que prevu) : leur probabilite augmente. Actions avec $A_t < 0$ : leur probabilite diminue.

**Value loss** : entraine le critic a predire $G_t$ par regression MSE. Le critic doit apprendre $V_\phi(s_t) \approx G_t$.

**Entropy bonus** : encourage l'exploration (coefficient 0.0 pour HalfCheetah, voir ci-dessus).

### Un seul optimiseur

Standard A2C : un seul optimiseur Adam avec les parametres de l'actor et du critic combines. La loss totale est minimisee conjointement. C'est plus simple que deux optimiseurs separes (comme dans REINFORCE+baseline).

### Gradient clipping

$\lVert {\max_{grad}} \rVert =0.5$ : on coupe les gradients dont la norme depasse 0.5. **Essentiel pour MuJoCo.** Sans ca, une grande recompense ou un grand avantage peut produire un gradient enorme qui fait exploser les poids. Le clipping stabilise l'entrainement au prix d'un apprentissage plus lent.

## Le probleme du biais-variance : peut-on faire mieux ?

### La limite des n-step returns

Le n-step return fixe $n$ a l'avance. Avec $n=2048$ et des episodes de 1000 pas, on accumule potentiellement 2048 recompenses avant de bootstrapper. La variance reste elevee.

Mais surtout : le choix de $n$ est arbitraire. Pourquoi 2048 et pas 512 ou 4096 ? Chaque valeur de $n$ donne un estimateur different, avec un profil biais-variance different. Choisir $n$ revient a choisir un point sur la courbe biais-variance.

**Intuition.** Le n-step return c'est comme demander l'avis d'un seul prof sur $n$ semestres. Avec $n=1$ (TD(0)), on ne fait confiance qu'au semestre suivant. Avec $n=\infty$ (Monte Carlo), on fait la moyenne sur toute la scolarite. Et si on pouvait demander l'avis de **tous les profs a la fois**, en donnant plus de poids aux plus recents ?

C'est exactement ce que propose GAE.

## GAE — Generalized Advantage Estimation

### L'idee : une moyenne ponderee de tous les estimateurs n-step

Plutot de choisir un seul $n$, GAE fait une **moyenne geometriquement ponderee** de tous les estimateurs d'avantage n-step. Les estimateurs courts (faible variance, fort biais) ont plus de poids, les longs (forte variance, faible biais) ont moins de poids.

Le parametre $\lambda \in [0, 1]$ controle la ponderation.

### La formule via les erreurs TD

On definit l'erreur TD au pas $t$ :

$$\delta_t = r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t)$$

C'est la "surprise" du critic : a-t-il correctement predit la valeur de l'etat $s_t$ en tenant compte de la recompense immediate $r_t$ et de la valeur de l'etat suivant ?

L'avantage GAE est alors :

$$\boxed{\hat{A}_t^{GAE(\gamma, \lambda)} = \sum_{l=0}^{T-t-1} (\gamma \lambda)^l \delta_{t+l}}$$

Ou, de facon recursive (efficace) :

$$\hat{A}_t^{GAE} = \delta_t + (\gamma \lambda)(1 - d_{t+1}) \hat{A}_{t+1}^{GAE}$$

### Cas limites

$$\begin{cases}
\lambda = 0 : & \hat{A}_t^{GAE} = \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t) \quad \text{(TD(0), fort biais)} \\
\lambda = 1 : & \hat{A}_t^{GAE} = \sum_{l \geq 0} \gamma^l \delta_{t+l} = G_t^{MC} - V(s_t) \quad \text{(Monte Carlo, forte variance)} \\
\lambda = 0.95 : & \text{sweet spot empirique (PPO, A2C, TRPO...)}
\end{cases}$$

**Analogie.** C'est comme une moyenne ponderee des notes de tous les profs jusqu'a la fin de l'annee. $\lambda$ controle combien on fait confiance aux profs des semestres lointains. Avec $\lambda=0$, on ecoute seulement le prof du semestre suivant. Avec $\lambda=1$, tous les profs ont le meme poids. Avec $\lambda=0.95$, les profs proches comptent beaucoup, les profs lointains de moins en moins.

In [ ]:
def compute_gae(rewards: np.ndarray, dones: np.ndarray, values: np.ndarray,
                last_value: float, gamma: float = 0.99,
                gae_lambda: float = 0.95) -> tuple:
    """Calcule les avantages GAE et les returns bootstrappes.
    
    Args:
        rewards    : (n_steps,) recompenses du rollout
        dones      : (n_steps,) flags de fin d'episode
        values     : (n_steps,) V(s_t) predit par le critic
        last_value : V(s_{t+n}) pour le bootstrap final
        gamma      : facteur d'actualisation
        gae_lambda : parametre GAE (0=TD(0), 1=Monte Carlo)
    
    Returns:
        advantages : (n_steps,) avantages GAE
        returns    : (n_steps,) cibles pour le critic (advantages + values)
    """
    n_steps = len(rewards)
    advantages = np.zeros(n_steps, dtype=np.float32)
    last_gae = 0.0

    # Etendre les valeurs avec le bootstrap final
    values_extended = np.append(values, last_value)  # (n_steps+1,)

    for t in reversed(range(n_steps)):
        # Erreur TD au pas t
        next_non_terminal = 1.0 - dones[t]
        delta = rewards[t] + gamma * values_extended[t + 1] * next_non_terminal - values_extended[t]
        # GAE recursif : accumulation ponderee des erreurs TD futures
        last_gae = delta + gamma * gae_lambda * next_non_terminal * last_gae
        advantages[t] = last_gae

    # Returns : cibles pour le critic = avantages + valeurs predites
    returns = advantages + values
    return advantages, returns


# --- Comparaison visuelle GAE vs N-step sur le meme rollout ---
np.random.seed(42)
n_demo = 50
rewards_cmp = np.random.randn(n_demo).astype(np.float32) * 0.5 + 0.3
dones_cmp = np.zeros(n_demo, dtype=np.float32)
dones_cmp[24] = 1.0  # un episode se termine au milieu
values_cmp = np.random.randn(n_demo).astype(np.float32) * 0.5  # critic imparfait
last_val = 0.5

# N-step returns -> advantages via soustraction
returns_nstep_cmp = compute_returns(rewards_cmp, dones_cmp, last_val, gamma=0.99)
advantages_nstep = returns_nstep_cmp - values_cmp

# GAE avec differents lambda
adv_gae_0, _ = compute_gae(rewards_cmp, dones_cmp, values_cmp, last_val, gamma=0.99, gae_lambda=0.0)
adv_gae_95, _ = compute_gae(rewards_cmp, dones_cmp, values_cmp, last_val, gamma=0.99, gae_lambda=0.95)
adv_gae_1, _ = compute_gae(rewards_cmp, dones_cmp, values_cmp, last_val, gamma=0.99, gae_lambda=1.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(advantages_nstep, color="steelblue", linewidth=2, label="N-step (A2C)", alpha=0.8)
ax.plot(adv_gae_95, color="coral", linewidth=2, label="GAE lambda=0.95 (A2C-GAE)", alpha=0.8)
ax.axvline(24, color="gray", linestyle="--", alpha=0.5, label="Fin d'episode")
ax.axhline(0, color="black", linestyle="-", linewidth=0.5, alpha=0.3)
ax.set_xlabel("Pas du rollout")
ax.set_ylabel("Avantage")
ax.set_title("Avantages : N-step vs GAE (lambda=0.95)")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(adv_gae_0, color="purple", linewidth=2, label="GAE lambda=0.0 (TD(0), fort biais)", alpha=0.8)
ax.plot(adv_gae_95, color="coral", linewidth=2, label="GAE lambda=0.95 (sweet spot)", alpha=0.8)
ax.plot(adv_gae_1, color="green", linewidth=2, label="GAE lambda=1.0 (MC, forte variance)", alpha=0.8)
ax.axvline(24, color="gray", linestyle="--", alpha=0.5, label="Fin d'episode")
ax.axhline(0, color="black", linestyle="-", linewidth=0.5, alpha=0.3)
ax.set_xlabel("Pas du rollout")
ax.set_ylabel("Avantage")
ax.set_title("GAE : effet de lambda sur les avantages")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Statistiques des avantages :")
print(f"  N-step (A2C)     : std={advantages_nstep.std():.3f}")
print(f"  GAE lambda=0.0   : std={adv_gae_0.std():.3f}  (TD(0), faible variance)")
print(f"  GAE lambda=0.95  : std={adv_gae_95.std():.3f}  (compromis)")
print(f"  GAE lambda=1.0   : std={adv_gae_1.std():.3f}  (MC, forte variance)")

## Resume des deux estimateurs cote a cote

Voici la seule ligne qui differe entre A2C et A2C-GAE. Tout le reste est identique.

| | A2C | A2C-GAE |
|-|-----|----------|
| Estimateur d'avantage | $A_t = G_t^{(n)} - V_\phi(s_t)$ | $A_t = \hat{A}_t^{GAE} = \sum_l (\gamma\lambda)^l \delta_{t+l}$ |
| Formule | `compute_returns` - valeurs | `compute_gae` |
| Hyperparametre extra | aucun | $\lambda$ (typiquement 0.95) |
| Biais | modere | reglable via $\lambda$ |
| Variance | moderee | generalement plus faible |

Les returns cibles pour le critic sont dans les deux cas : $G_t = A_t + V_\phi(s_t)$.

**La seule difference d'implementation** : la methode `_compute_advantages()` de l'agent.

In [ ]:
class A2CAgent:
    """Agent A2C avec n-step returns.
    
    Methode _compute_advantages : surchargee par A2CGAEAgent.
    """

    def __init__(self,
                 obs_dim: int,
                 action_dim: int,
                 hidden_dim: int = 256,
                 lr: float = 3e-4,
                 gamma: float = 0.99,
                 n_steps: int = 2048,
                 value_coef: float = 0.5,
                 entropy_coef: float = 0.0,
                 max_grad_norm: float = 0.5):
        self.gamma = gamma
        self.n_steps = n_steps
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm

        self.actor = GaussianActor(obs_dim, action_dim, hidden_dim)
        self.critic = CriticNetwork(obs_dim, hidden_dim)
        self.buffer = RolloutBuffer(n_steps, obs_dim, action_dim)

        self.optimizer = optim.Adam(
            list(self.actor.parameters()) + list(self.critic.parameters()),
            lr=lr,
            eps=1e-5,
        )

        self.history = {"rewards": [], "policy_loss": [], "value_loss": [], "entropy": []}
        self._current_obs = None

    def _compute_advantages(self, rewards, dones, values, last_value):
        """Calcule les avantages et les returns cibles pour le critic.
        
        Version A2C : n-step returns bootstrappes.
        Surchargee dans A2CGAEAgent avec compute_gae.
        """
        returns = compute_returns(rewards, dones, last_value, self.gamma)
        advantages = returns - values
        return advantages, returns

    def _update(self, last_obs: np.ndarray) -> dict:
        """Effectue une mise a jour de politique sur le rollout courant."""
        with torch.no_grad():
            last_obs_t = torch.tensor(last_obs, dtype=torch.float32).unsqueeze(0)
            last_value = self.critic(last_obs_t).item()

        obs_t, actions_t, rewards_np, dones_np, values_np, _ = self.buffer.get_tensors()

        advantages_np, returns_np = self._compute_advantages(
            rewards_np.numpy(),
            dones_np.numpy(),
            values_np.numpy(),
            last_value,
        )
        advantages_t = torch.tensor(advantages_np, dtype=torch.float32)
        returns_t = torch.tensor(returns_np, dtype=torch.float32)

        advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)

        # On re-evalue la log-probabilite des ACTIONS BRUTES tirees par la politique.
        # L'action clippee sert seulement a interagir avec Gym, pas a calculer le gradient.
        log_probs, entropy = self.actor.evaluate(obs_t, actions_t)
        values_pred = self.critic(obs_t)

        policy_loss = -(log_probs * advantages_t).mean()
        value_loss = F.mse_loss(values_pred, returns_t)
        entropy_loss = -entropy.mean()

        total_loss = policy_loss + self.value_coef * value_loss + self.entropy_coef * entropy_loss

        self.optimizer.zero_grad()
        total_loss.backward()
        nn.utils.clip_grad_norm_(
            list(self.actor.parameters()) + list(self.critic.parameters()),
            self.max_grad_norm,
        )
        self.optimizer.step()

        return {
            "policy_loss": policy_loss.item(),
            "value_loss": value_loss.item(),
            "entropy": -entropy_loss.item(),
        }

    def learn_step(self, env) -> dict:
        """Collecte n_steps transitions et effectue une mise a jour."""
        self.buffer.reset()

        if self._current_obs is None:
            obs, _ = env.reset()
        else:
            obs = self._current_obs

        episode_rewards = []
        current_ep_reward = 0.0

        for _ in range(self.n_steps):
            with torch.no_grad():
                policy_action, log_prob = self.actor.get_action(obs)
                obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
                value = self.critic(obs_t).item()

            # Deux actions differentes :
            # - policy_action : action brute echantillonnee par pi_theta, coherente avec log_prob ;
            # - env_action    : action clippee envoyee a Gym, car HalfCheetah impose [-1, 1].
            env_action = np.clip(policy_action, env.action_space.low, env.action_space.high)

            next_obs, reward, terminated, truncated, _ = env.step(env_action)
            done = terminated or truncated
            env_reward = float(reward)
            stored_reward = env_reward

            # Truncation value bootstrap (pattern SB3 (https://github.com/DLR-RM/stable-baselines3) / Pardo et al.).
            # On coupe bien la trajectoire au time limit, mais on ajoute gamma * V(s_T)
            # pour ne pas traiter cette coupure artificielle comme un terminal de valeur zero.
            if truncated and not terminated:
                with torch.no_grad():
                    obs_t_trunc = torch.tensor(next_obs, dtype=torch.float32).unsqueeze(0)
                    terminal_value = self.critic(obs_t_trunc).item()
                stored_reward = env_reward + self.gamma * terminal_value

            self.buffer.add(
                obs=obs,
                action=policy_action,          # important : action brute, pas action clippee
                reward=stored_reward,
                done=float(done),
                value=value,
                log_prob=log_prob.item(),
            )

            current_ep_reward += env_reward
            obs = next_obs

            if done:
                episode_rewards.append(current_ep_reward)
                current_ep_reward = 0.0
                obs, _ = env.reset()

        self._current_obs = obs
        episode_rewards.append(current_ep_reward)

        metrics = self._update(obs)
        metrics["mean_reward"] = np.mean(episode_rewards) if episode_rewards else 0.0

        for key in ["rewards", "policy_loss", "value_loss", "entropy"]:
            self.history[key].append(
                metrics.get("mean_reward" if key == "rewards" else key, 0.0)
            )

        return metrics

In [ ]:
# --- Test rapide A2C ---
torch.manual_seed(42)
agent_test = A2CAgent(obs_dim=17, action_dim=6)
print("A2CAgent cree :")
print(f"  Actor  : {sum(p.numel() for p in agent_test.actor.parameters()):,} parametres")
print(f"  Critic : {sum(p.numel() for p in agent_test.critic.parameters()):,} parametres")
print(f"  Buffer : {agent_test.n_steps} steps")

### Le piege terminated vs truncated : truncation value bootstrap

#### Le probleme

Gymnasium distingue deux types de fin d'episode :

- **`terminated`** : l'agent a atteint un etat terminal — le robot est tombe, le jeu est fini. La valeur future est zero par definition.
- **`truncated`** : la limite de temps a ete atteinte (1000 steps dans HalfCheetah). L'episode s'arrete, mais le monde continue — si l'agent avait pu continuer, il aurait accumule des rewards supplementaires.

HalfCheetah ne termine jamais naturellement : les episodes sont **toujours tronques**, jamais termines. Or, `compute_returns` utilise le masque `done` pour couper le bootstrap :

$$G_t = r_t + \gamma \cdot (1 - \text{done}_t) \cdot V(s_{t+1})$$

Quand `done=1`, on coupe — on suppose que la valeur future est nulle. Si on marque les truncations comme `done=1` sans compensation, on sous-estime systematiquement les returns de chaque episode.

#### La fausse solution

Une premiere intuition : marquer `done = float(terminated)` plutot que `done = float(terminated or truncated)`. On ne coupe le bootstrap que pour les vrais episodes terminaux. Cela semble correct — mais cela introduit un bug plus subtil.

Le buffer stocke des transitions sequentielles. Quand `done=0` au moment du reset, `compute_returns` chaine les returns a travers la frontiere d'episode : le reward du dernier step de l'episode $k$ est bootstrap par la valeur du premier step de l'episode $k+1$. Deux trajectoires independantes sont melangees, et les returns sont corrompus.

#### La bonne solution : le truncation value bootstrap (pattern SB3)

La solution correcte, utilisee par Stable-Baselines3 intoduit dans ce papier [Pardo et al.,  (ICML/PMLR 2018)](https://arxiv.org/abs/1712.00378), discuter [ici](https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html) , gere les trois cas separement :

| Situation | `done` stocke | Reward modifie |
|-----------|---------------|----------------|
| Episode termine (`terminated=True`) | `1.0` | Inchange |
| Episode tronque (`truncated=True, terminated=False`) | `1.0` | `reward += gamma * V(s_T)` |
| Pas de fin d'episode | `0.0` | Inchange |

- `done=1` coupe le chainage dans `compute_returns` — on ne melange jamais deux trajectoires.
- `reward += gamma * V(s_T)` compense la valeur future perdue : on ajoute au dernier reward l'estimation de ce que l'agent aurait accumule s'il avait continue.

```python
if truncated and not terminated:
    terminal_value = critic(next_obs).item()
    reward += gamma * terminal_value
buffer.add(..., done=float(done), ...)  # done = terminated or truncated, coupe le chainage
```

L'effet est identique a un episode infini dont on a seulement observe une fenetre : la valeur terminale sert de "relai" entre la fenetre observee et le futur non observe.

Ce detail d'implementation est critique sur HalfCheetah. Sans lui, les returns sont systematiquement tronques a zero en fin de chaque episode, le critic apprend a sous-estimer $V(s)$, et la politique recoit des gradients biaises vers des comportements courts. Le score greedy tombe a ~0 au lieu de ~40-50 apres 200k steps.

In [ ]:
class A2CGAEAgent(A2CAgent):
    """Agent A2C avec GAE (Generalized Advantage Estimation).
    
    Seule difference avec A2CAgent : _compute_advantages utilise compute_gae.
    Tout le reste (actor, critic, optimizer, buffer, _update, learn_step) est herite.
    """

    def __init__(self, *args, gae_lambda: float = 0.95, **kwargs):
        super().__init__(*args, **kwargs)
        self.gae_lambda = gae_lambda

    def _compute_advantages(self, rewards, dones, values, last_value):
        """Override : utilise GAE au lieu des n-step returns bruts."""
        advantages, returns = compute_gae(
            rewards, dones, values, last_value,
            gamma=self.gamma, gae_lambda=self.gae_lambda
        )
        return advantages, returns

In [ ]:
# --- Verification : seule _compute_advantages differe ---
print("Methodes surchargees dans A2CGAEAgent :")
for name in dir(A2CGAEAgent):
    if not name.startswith("__"):
        base_method = getattr(A2CAgent, name, None)
        child_method = getattr(A2CGAEAgent, name, None)
        if callable(base_method) and callable(child_method):
            if child_method is not base_method:
                print(f"  -> {name} (surcharge)")

torch.manual_seed(42)
agent_gae_test = A2CGAEAgent(obs_dim=17, action_dim=6, gae_lambda=0.95)
print(f"\nA2CGAEAgent cree (lambda={agent_gae_test.gae_lambda}) :")
print(f"  Actor  : {sum(p.numel() for p in agent_gae_test.actor.parameters()):,} parametres (identique)")
print(f"  Critic : {sum(p.numel() for p in agent_gae_test.critic.parameters()):,} parametres (identique)")

In [ ]:
# --- Demo : meme rollout, avantages differents ---
np.random.seed(42)
n_demo = 20
rewards_d = np.random.randn(n_demo).astype(np.float32) * 0.5 + 0.3
dones_d = np.zeros(n_demo, dtype=np.float32)
values_d = np.zeros(n_demo, dtype=np.float32) + 1.0  # critic constant (demo)
last_v = 1.0

# A2C : n-step
returns_a2c = compute_returns(rewards_d, dones_d, last_v, gamma=0.99)
advantages_a2c = returns_a2c - values_d

# A2C-GAE
advantages_gae, returns_gae = compute_gae(rewards_d, dones_d, values_d, last_v,
                                           gamma=0.99, gae_lambda=0.95)

print("Comparaison avantages sur le meme rollout (20 pas) :")
print(f"{'Pas':>4} | {'Reward':>8} | {'Adv A2C':>10} | {'Adv GAE':>10}")
print("-" * 42)
for i in range(n_demo):
    print(f"{i:>4} | {rewards_d[i]:>8.4f} | {advantages_a2c[i]:>10.4f} | {advantages_gae[i]:>10.4f}")

print(f"\nStd avantages A2C : {advantages_a2c.std():.4f}")
print(f"Std avantages GAE : {advantages_gae.std():.4f}")
print(f"\nGAE lisse les avantages -> variance reduite de {100*(1 - advantages_gae.std()/advantages_a2c.std()):.1f}%")

## Les deux algorithmes complets

$$\boxed{\begin{aligned}
&\textbf{Algorithme : A2C / A2C-GAE} \\
&\text{Initialisation : actor } \pi_\theta,\ \text{critic } V_\phi,\ \text{optimiseur Adam},\ \text{buffer de taille } n_{steps} \\
\\
&\textbf{Pour chaque iteration (jusqu'a total\_timesteps) :} \\
&\quad \text{1. Collecter } n_{steps} \text{ transitions} : (s_t, a_t, r_t, d_t, V_\phi(s_t), \log\pi_\theta(a_t|s_t)) \\
&\quad \text{2. Bootstrap : } last\_value = V_\phi(s_{t+n}) \\
&\quad \text{3. Calculer les avantages :} \\
&\quad\quad \textbf{[A2C]}\quad A_t = \underbrace{\sum_{k=0}^{n-1}\gamma^k r_{t+k} + \gamma^n V(s_{t+n})}_{G_t^{(n)}} - V_\phi(s_t) \\
&\quad\quad \textbf{[A2C-GAE]}\quad A_t = \hat{A}_t^{GAE} = \sum_{l \geq 0} (\gamma\lambda)^l \delta_{t+l},\ \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t) \\
&\quad \text{4. Normaliser : } A_t \leftarrow (A_t - \bar{A}) / (\sigma_A + \epsilon) \\
&\quad \text{5. Calculer les returns cibles : } G_t = A_t + V_\phi(s_t) \\
&\quad \text{6. Loss totale :} \\
&\quad\quad \mathcal{L} = \underbrace{-\frac{1}{n}\sum_t \log\pi_\theta(a_t|s_t) \cdot A_t}_{\text{policy}} + \underbrace{c_V \cdot \frac{1}{n}\sum_t (V_\phi(s_t) - G_t)^2}_{\text{value}} \\
&\quad \text{7. Backprop + gradient clipping + Adam step} \\
&\quad \text{8. Vider le buffer}
\end{aligned}}$$

## Hyperparametres

On fixe **exactement les memes valeurs** pour les deux algorithmes afin d'isoler l'effet de GAE. La seule difference est $\lambda$.

| Parametre | Valeur | Role |
|-----------|--------|------|
| `hidden_dim` | 256 | Taille des couches cachees (standard MuJoCo) |
| `lr` | 3e-4 | Taux d'apprentissage Adam |
| `gamma` | 0.99 | Facteur d'actualisation |
| `n_steps` | 512 | Longueur du rollout utilisee ici : plus courte que 2048 pour obtenir davantage d'updates visibles sur 200k pas, sans changer la formule de l'algorithme |
| `value_coef` | 0.5 | Poids de la loss du critic |
| `entropy_coef` | 0.0 | Bonus d'entropie laisse nul dans cette demonstration |
| `max_grad_norm` | 0.5 | Clipping du gradient (essentiel MuJoCo) |
| `total_timesteps` | 200_000 | Duree de l'entrainement |
| `gae_lambda` | 0.95 | Parametre GAE (A2C-GAE uniquement) |
| `seed` | 42 | Graine aleatoire (identique pour les deux) |

**Note.** 200 000 pas suffit pour rendre visibles les differences de comportement sur ce notebook. Cela ne remplace ni une etude multi-seed ni un entrainement long de reference.

In [ ]:
def train_agent(agent, env, total_timesteps, log_interval, label):
    """Boucle d'entrainement generique (A2C ou A2C-GAE)."""
    print(f"{'=' * 60}")
    print(f"ENTRAINEMENT {label}")
    print(f"{'=' * 60}")
    timesteps = 0
    iteration = 0
    last_log_t = 0

    while timesteps < total_timesteps:
        metrics = agent.learn_step(env)
        timesteps += agent.n_steps
        iteration += 1

        if timesteps - last_log_t >= log_interval or timesteps >= total_timesteps:
            last_log_t = timesteps
            print(
                f"Step {timesteps:>7d} | "
                f"Reward={metrics['mean_reward']:>8.1f} | "
                f"pi_loss={metrics['policy_loss']:>8.4f} | "
                f"v_loss={metrics['value_loss']:>8.4f} | "
                f"entropy={metrics['entropy']:>6.3f}"
            )

    print(f"\n{label} termine ({total_timesteps} steps).")

In [ ]:
# Configuration commune des deux expériences A2C.
SEED = 42
ENV_ID = "HalfCheetah-v5"
HIDDEN_DIM = 256
LR = 3e-4
GAMMA = 0.99
N_STEPS = 512
VALUE_COEF = 0.5
ENTROPY_COEF = 0.0
MAX_GRAD_NORM = 0.5
TOTAL_TIMESTEPS = 200_000
GAE_LAMBDA = 0.95
LOG_INTERVAL = 10_000

torch.manual_seed(SEED)
np.random.seed(SEED)


In [ ]:
# === Entrainement A2C ===
torch.manual_seed(SEED)
np.random.seed(SEED)

env_a2c = gym.make(ENV_ID)
env_a2c.reset(seed=SEED)

obs_dim = env_a2c.observation_space.shape[0]
action_dim = env_a2c.action_space.shape[0]

agent_a2c = A2CAgent(
    obs_dim=obs_dim,
    action_dim=action_dim,
    hidden_dim=HIDDEN_DIM,
    lr=LR,
    gamma=GAMMA,
    n_steps=N_STEPS,
    value_coef=VALUE_COEF,
    entropy_coef=ENTROPY_COEF,
    max_grad_norm=MAX_GRAD_NORM,
)

train_agent(agent_a2c, env_a2c, TOTAL_TIMESTEPS, LOG_INTERVAL, "A2C")
env_a2c.close()

In [ ]:
# === Entrainement A2C-GAE ===
torch.manual_seed(SEED)
np.random.seed(SEED)

env_gae = gym.make(ENV_ID)
env_gae.reset(seed=SEED)

agent_gae = A2CGAEAgent(
    obs_dim=obs_dim,
    action_dim=action_dim,
    hidden_dim=HIDDEN_DIM,
    lr=LR,
    gamma=GAMMA,
    n_steps=N_STEPS,
    value_coef=VALUE_COEF,
    entropy_coef=ENTROPY_COEF,
    max_grad_norm=MAX_GRAD_NORM,
    gae_lambda=GAE_LAMBDA,
)

train_agent(agent_gae, env_gae, TOTAL_TIMESTEPS, LOG_INTERVAL, "A2C-GAE")
env_gae.close()

In [ ]:
# === Courbes comparatives ===
def moving_average(data, window=5):
    if len(data) < window:
        return np.array(data)
    return np.convolve(data, np.ones(window) / window, mode="valid")


fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("A2C vs A2C-GAE sur HalfCheetah-v5 (200k steps)", fontsize=14, fontweight="bold")

# Axe des x : iterations (chaque iteration = n_steps timesteps)
n_iters_a2c = len(agent_a2c.history["rewards"])
n_iters_gae = len(agent_gae.history["rewards"])
x_a2c = np.arange(n_iters_a2c) * N_STEPS / 1000  # en milliers de steps
x_gae = np.arange(n_iters_gae) * N_STEPS / 1000
window = 3

keys = ["rewards", "policy_loss", "value_loss", "entropy"]
titles = ["Recompense moyenne", "Policy Loss", "Value Loss (MSE)", "Entropie"]
colors_a2c, colors_gae = "steelblue", "coral"

for ax, key, title in zip(axes.flat, keys, titles):
    data_a2c = agent_a2c.history[key]
    data_gae = agent_gae.history[key]

    ax.plot(x_a2c, data_a2c, alpha=0.2, color=colors_a2c)
    ax.plot(x_gae, data_gae, alpha=0.2, color=colors_gae)

    ma_a2c = moving_average(data_a2c, window)
    ma_gae = moving_average(data_gae, window)
    x_ma_a2c = x_a2c[window - 1:window - 1 + len(ma_a2c)]
    x_ma_gae = x_gae[window - 1:window - 1 + len(ma_gae)]

    ax.plot(x_ma_a2c, ma_a2c, color=colors_a2c, linewidth=2, label="A2C (n-step)")
    ax.plot(x_ma_gae, ma_gae, color=colors_gae, linewidth=2, label=f"A2C-GAE ($\\lambda$=0.95)")

    ax.set_xlabel("Timesteps (x1000)")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Resume numerique
print("\nResume numerique (20 dernieres iterations) :")
print(f"{'Metrique':>20} | {'A2C':>12} | {'A2C-GAE':>12}")
print("-" * 50)
for key in keys:
    v_a2c = np.mean(agent_a2c.history[key][-20:])
    v_gae = np.mean(agent_gae.history[key][-20:])
    print(f"{key:>20} | {v_a2c:>12.4f} | {v_gae:>12.4f}")

### Interpretation des resultats : pourquoi A2C apprend et A2C-GAE peut s'effondrer ici

#### A2C : apprentissage stable mais lent sur cette fenetre

- Recompense greedy ~43 : l'agent a appris a avancer (vs ~0 sans le truncation bootstrap).
- Value loss descend de 727 a ~100 : le critic apprend les returns correctement.
- Entropie autour de ~8.5 sur ce run : avec `entropy_coef=0`, le `log_std` n'est pas directement regularise. L'exploration vient donc surtout de l'ecart-type appris par la gaussienne.
- 200k steps = ~390 updates seulement avec `n_steps=512`. On voit deja la dynamique, mais pas encore un regime asymptotique stable.

#### A2C-GAE : precision puis effondrement sur ce run

- **Steps 0-130k** : apprentissage comparable a A2C, avec une value loss plus basse — signe que le critic ajuste plus vite ses cibles.
- **Step 153k** : la politique commence a produire des rewards positifs pour la premiere fois.
- **Step 163k** : les avantages deviennent grands positifs, entrainant une mise a jour massive de la politique (`pi_loss = -10.26`).
- **Step 174k** : la politique a trop bouge, les actions deviennent mauvaises, les avantages passent fortement negatifs, correction massive en sens inverse (`pi_loss = +25.9`).
- Cette spirale d'oscillations apparait **dans cette configuration et sur cette seed**. Elle illustre bien le probleme on-policy, sans prouver qu'un effondrement identique est inevitable sur toutes les variantes d'A2C-GAE.

#### Pourquoi GAE peut devenir plus fragile ici ?

GAE ($\lambda=0.95$) donne des estimations d'avantage plus lisses que les n-step returns bruts. Quand le critic est bon, c'est un avantage net. Mais cette precision rend aussi l'update plus sensible a un critic qui commence a deriver : les avantages changent alors de signe et de magnitude de facon plus structuree. Sans mecanisme de protection sur la politique, une seule grosse mise a jour peut suffire a envoyer l'acteur dans une mauvaise region.

A2C vanilla est parfois plus tolerant dans ces conditions : son estimateur d'avantage est plus bruite, ce qui peut amortir certaines transitions brusques. Ce n'est pas une garantie de robustesse, plutot un effet secondaire du signal plus rugueux.

#### Le probleme que PPO cherche a resoudre

C'est exactement le probleme que **PPO** (Proximal Policy Optimization, [Schulman et al., 2017](https://arxiv.org/abs/1707.06347)) cherche a limiter. PPO introduit un ratio de clipping qui borne la taille de chaque mise a jour de politique :

$$r(\theta) = \frac{\pi_\theta(a|s)}{\pi_{\theta_\text{old}}(a|s)}$$

La loss PPO clippe ce ratio dans $[1-\epsilon,\; 1+\epsilon]$ (typiquement $\epsilon = 0.2$), empechant la politique de s'eloigner trop en une seule mise a jour :

$$\mathcal{L}^\text{PPO} = \mathbb{E}_t\!\left[\min\!\left(r_t(\theta)\,\hat{A}_t,\; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\,\hat{A}_t\right)\right]$$

PPO ne rend pas GAE magique; il ajoute un garde-fou quand les avantages poussent trop fort. Ce sera l'objet du notebook suivant.

## Analyse des courbes

### Courbe de recompense

Les deux algorithmes apprennent a faire courir HalfCheetah vers la droite, mais pas avec la meme regularite sur ce mini-training :

- **A2C** : progression reguliere mais avec des oscillations marquees. Les n-step returns peuvent donner des estimations plus rugueuses de l'avantage, surtout en debut d'entrainement quand le critic est encore imparfait.
- **A2C-GAE** : depart souvent plus propre, puis comportement plus sensible quand le critic et la politique se desalignent. Sur ce run, cette sensibilite finit par dominer.

### Policy loss

La policy loss ne mesure pas directement la qualite de la politique. Elle peut etre negative (les log-probabilites des bonnes actions augmentent) ou positive. Ce qui compte : elle doit rester interpretable et ne pas partir en oscillations geantes. Ici, c'est justement l'un des signaux qui annonce la degradation d'A2C-GAE.

### Value loss (MSE)

La value loss doit **decroitre** : le critic apprend progressivement a predire les returns. Une value loss qui reste elevee = un critic imprecis = des avantages mal estimes = un signal de gradient bruite. GAE peut accelerer cet apprentissage tant que le critic reste bien calibre.

### Entropie

Avec `entropy_coef=0.0`, l'entropie n'est pas directement optimisee. Elle evolue donc comme un thermometre passif de la dispersion de la politique : elle peut baisser quand l'acteur se specialise, ou rester haute si le `log_std` appris ne se contracte pas vraiment.

### Tableau recapitulatif

| Dimension | A2C | A2C-GAE |
|-----------|-----|----------|
| Estimateur | n-step bootstrappe | GAE exponentiel |
| Biais | modere, fixe | reglable via $\lambda$ |
| Variance | moderee | souvent plus faible au debut |
| Parametre extra | aucun | $\lambda = 0.95$ |
| Stabilite sur ce run | reguliere mais limitee | plus sensible apres un bon depart |
| Vitesse de convergence | correcte | souvent meilleure tant que le critic reste fiable |
| Complexite code | de base | +5 lignes (compute_gae) |
| Usage dans la litterature | courant | standard (PPO, TRPO) |

In [ ]:
# === Démonstration finale : replay vidéo des politiques déterministes ===
def evaluate_and_display_agent(
    agent,
    label,
    n_episodes=5,
    seed=0,
    video_root="videos/04_a2c_halfcheetah",
):
    """Évalue la politique déterministe et affiche le dernier replay vidéo enregistré."""
    rewards = []
    safe_label = label.lower().replace(" ", "_").replace("-", "_")
    video_dir = Path(video_root) / safe_label
    video_dir.mkdir(parents=True, exist_ok=True)

    env = gym.make(ENV_ID, render_mode="rgb_array")
    env = RecordVideo(
        env,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == n_episodes - 1,
        name_prefix=f"{safe_label}_greedy",
    )

    agent.actor.eval()

    print(f"{'=' * 50}")
    print(f"Démo greedy : {label}")
    print(f"{'=' * 50}")

    try:
        for ep in range(n_episodes):
            obs, _ = env.reset(seed=seed + ep)
            total = 0.0
            steps = 0
            done = False

            while not done:
                with torch.no_grad():
                    obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
                    dist = agent.actor(obs_t)
                    action = dist.mean.squeeze(0).numpy()
                    action = np.clip(action, env.action_space.low, env.action_space.high)

                obs, reward, terminated, truncated, _ = env.step(action)
                total += reward
                steps += 1
                done = terminated or truncated

            rewards.append(total)
            print(f"[{label}] Épisode {ep + 1} : {total:.1f} (en {steps} pas)")

    finally:
        env.close()

    print(f"[{label}] Moyenne : {np.mean(rewards):.1f} +/- {np.std(rewards):.1f}")

    videos = sorted(video_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
    if videos and Video is not None and display is not None:
        print(f"Replay vidéo {label} :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=520))
    elif not videos:
        print(f"Aucune vidéo générée pour {label}.")
    else:
        print(f"Vidéo générée mais affichage IPython indisponible : {videos[-1]}")

    return rewards


demo_a2c = evaluate_and_display_agent(agent_a2c, "A2C")
demo_gae = evaluate_and_display_agent(agent_gae, "A2C-GAE")

print("Comparaison finale (5 épisodes greedy) :")
print(f"  A2C     : {np.mean(demo_a2c):.1f} +/- {np.std(demo_a2c):.1f}")
print(f"  A2C-GAE : {np.mean(demo_gae):.1f} +/- {np.std(demo_gae):.1f}")


## Au-dela du mini-training : instabilite et catastrophic forgetting

Les resultats ci-dessus montrent un entrainement court (200k steps). Que se passe-t-il quand on entraine plus longtemps, sur 1 million de pas ?

### Ce qu'on observe sur les runs complets de ce chapitre

Sur les runs longs associes a ce notebook :

- **A2C** apprend progressivement, atteint un pic d'evaluation autour de 500, puis peut **collapse** : le meilleur checkpoint (`best.pt`) reste nettement meilleur que le modele final.
- **A2C-GAE** peut se degrader plus tot et plus violemment sur certaines seeds, avec une policy loss qui part en oscillations et des recompenses qui chutent.
- Ces trajectoires ne prouvent pas une impossibilite generale d'A2C ou d'A2C-GAE; elles montrent qu'avec cette recette simple, l'absence de garde-fou sur la politique finit par couter cher.

### Trois causes qui se combinent

**1. Aucune contrainte sur la taille des mises a jour**

A2C optimise directement :

$$\mathcal{L}_{policy} = -\frac{1}{n} \sum_t \log \pi_\theta(a_t|s_t) \cdot A_t$$

Si les avantages $A_t$ sont grands (dans un sens ou dans l'autre), la politique peut changer radicalement en un seul update. Il n'y a ni ratio clipping (comme PPO), ni contrainte KL (comme TRPO), ni rollback. Un seul mauvais update peut suffire a declencher une spirale : mauvaise politique $\to$ mauvaises trajectoires $\to$ mauvais avantages $\to$ update encore pire.

**2. Le critic peut devenir instable**

Quand l'**explained variance** devient negative, le critic fait *pire* que predire la moyenne des returns. Les avantages -- surtout avec GAE, qui depend finement du critic -- deviennent du bruit *structure* qui guide la politique dans la mauvaise direction. C'est plus pernicieux que du bruit aleatoire : GAE produit des signaux "propres" mais faux.

Dans les runs longs associes a ce chapitre, l'explained variance d'A2C-GAE finit par devenir negative sur les trajectoires qui se degradent, ce qui renforce cette lecture.

**3. Actions hors bornes et politique gaussienne**

Les politiques gaussiennes peuvent echantillonner des actions hors des bornes de l'environnement. Par exemple, la politique produit $a = 3.0$, mais HalfCheetah attend $a \in [-1, 1]$. L'action envoyee a Gym est clippee a $1.0$, mais $\log \pi(a = 3.0 | s)$ est calcule sur l'action brute. Cette incoherence corrompt subtilement le gradient de politique.

La solution correcte : stocker l'action brute (coherente avec `log_prob`) dans le buffer, et n'envoyer l'action clippee qu'a l'environnement. A plus long terme, la solution propre est la **tanh-squashed Gaussian** (utilisee par SAC), qui produit des actions toujours dans les bornes par construction.

### Pourquoi PPO aide souvent dans ce probleme

PPO ajoute un mecanisme de freinage explicite. Au lieu d'optimiser directement $\log \pi \cdot A$, il utilise un **ratio clippe** :

$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}, \quad \mathcal{L}^{CLIP} = \mathbb{E}_t \left[ \min\left(r_t A_t, \; \text{clip}(r_t, 1-\varepsilon, 1+\varepsilon) \, A_t \right) \right]$$

Meme si l'avantage pousse fort, le clipping empeche la politique de bouger trop loin de $\pi_{old}$ en un seul update. En pratique, sur ces runs, cela suffit souvent a conserver une politique exploitable plus longtemps que la version A2C nue.

### Pont vers SAC et les methodes off-policy

SAC (Soft Actor-Critic) va encore plus loin en combinant trois idees :

- **tanh-squashed Gaussian** : les actions sont toujours dans les bornes par construction, sans clipping ad hoc;
- **Replay buffer** : off-policy, chaque transition peut etre reutilisee plusieurs fois, ce qui ameliore considerablement l'efficacite d'echantillonnage;
- **Entropie maximale** : la politique est entrainee a maximiser la recompense ET l'entropie, ce qui maintient l'exploration tout au long de l'entrainement.

Ce sera l'objet des notebooks suivants.

### Lecon pratique

Pour A2C et A2C-GAE sur du controle continu :

1. Toujours conserver `best.pt` comme modele de reference, pas le dernier checkpoint.
2. Suivre les diagnostics pendant l'entrainement : `explained_variance` (si negative, le critic a decroche), `grad_norm` (si constamment au max, le gradient clipping sature), `log_std` (si il collapse, l'exploration s'est arretee), `action_clip_fraction` (si elevee, les actions sortent des bornes).
3. Considerer ces algorithmes comme des **outils pedagogiques** pour comprendre actor-critic. Pour un training stable sur controle continu, privilegier PPO (on-policy) ou SAC (off-policy).

## Conclusion et ponts vers la suite

### Ce qu'on a construit

Deux algorithmes **actor-critic** sur un vrai environnement de controle continu :

- **A2C** : collecte un rollout de $n$ pas, calcule les n-step returns bootstrappes, met a jour actor et critic avec une seule loss. Plus rapide et stable que REINFORCE grace au bootstrapping.
- **A2C-GAE** : identique, sauf que l'avantage est calcule par GAE : une moyenne ponderee exponentiellement de toutes les erreurs TD futures. Le parametre $\lambda$ permet de regler le compromis biais-variance en continu.

Les briques introduites dans ce notebook :
- **GaussianActor** : politique gaussienne pour actions continues. $\log\sigma$ appris comme parametre.
- **RolloutBuffer** : buffer on-policy pre-alloue, vide apres chaque mise a jour.
- **compute_gae** : estimateur GAE, standard dans tous les algorithmes modernes (PPO, TRPO, A3C).
- **Gradient clipping** : essentiel pour MuJoCo, stabilise l'entrainement.

### GAE : le standard de facto

GAE est utilise dans pratiquement tous les algorithmes policy gradient modernes :
- **PPO** (Proximal Policy Optimization) : GAE + clipping du ratio de probabilites
- **TRPO** (Trust Region) : GAE + contrainte KL
- **A3C** (Asynchronous A3C) : GAE distribue sur plusieurs workers
- **SAC** (Soft Actor-Critic) : similaire en esprit, version off-policy

### Pont vers A3C

On a construit un agent synchrone : un seul environnement, une seule boucle de collecte. Pour **paralleliser** (A3C) :
- $N$ workers, chacun avec son propre environnement et sa copie locale du modele
- Chaque worker collecte son rollout, calcule ses gradients, et les pousse vers un modele global partage (`torch.multiprocessing` + `shared_memory`)
- Les methodes `_compute_advantages()` et `_update()` qu'on a ecrites sont **directement reutilisables** dans chaque worker
- La difference cle : les gradients sont calcules localement et appliques au modele global de facon asynchrone

### Pont vers PPO

PPO resout un probleme d'A2C : un seul gradient step par rollout est peu efficace. PPO fait $K$ epochs sur le meme rollout en clippant le ratio :

$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}, \quad \mathcal{L}^{CLIP} = \mathbb{E}_t\left[\min(r_t A_t,\ \text{clip}(r_t, 1-\varepsilon, 1+\varepsilon) A_t)\right]$$

Le clipping empeche des mises a jour trop grandes, permettant plusieurs passes sur le meme rollout sans destabiliser la politique.

### Pont vers l'off-policy

A2C est on-policy : chaque transition n'est utilisee qu'une fois. DDPG, TD3, SAC utilisent un **replay buffer** et peuvent reutiliser les transitions de politiques anterieures. SAC ajoute une entropie maximale et devient l'etat de l'art sur les environnements continus complexes.

---

**References**
- Mnih, V. et al. (2016). Asynchronous Methods for Deep Reinforcement Learning. [arXiv:1602.01783](https://arxiv.org/abs/1602.01783). (A3C, dont A2C est la version synchrone)
- Schulman, J. et al. (2016). High-Dimensional Continuous Control Using Generalized Advantage Estimation. [arXiv:1506.02438](https://arxiv.org/abs/1506.02438). (GAE)
- Schulman, J. et al. (2017). Proximal Policy Optimization Algorithms. [arXiv:1707.06347](https://arxiv.org/abs/1707.06347). (PPO, utilise GAE)
- Todorov, E., Erez, T. & Tassa, Y. (2012). MuJoCo: A physics engine for model-based control. [IROS 2012](https://ieeexplore.ieee.org/document/6386109).